In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from multiprocessing import Pool

In [ ]:
# CONFIG
LEAD_TIME = 1
INPUT_DIR = "/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa"
OUTPUT_DIR = f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa_sharded{LEAD_TIME}"


In [3]:
# Shard size: you can adjust this
SAMPLES_PER_SHARD = 1000

In [4]:
# Create output dir
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
# Load file list (you can use your train/test/val csv)
file_list = pd.read_csv('/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/splits/train_files.csv', header=None)[0].tolist()

In [10]:
def process_one(idx):
    """
    Load one sample and return (input_tensor, target_tensor)
    """
    fname = file_list[idx]
    
    input_path = os.path.join(INPUT_DIR, "inputs_t0", fname)
    timestamp = fname.replace("input-", "")
    target_fname = f"target-{timestamp}"
    target_path = os.path.join(INPUT_DIR, f"targets_t{LEAD_TIME}", target_fname)

    inputs = torch.load(input_path).float()
    targets = torch.load(target_path).float()  # (1024, 1024)
    return (inputs, targets)

In [11]:
# Loop through shards
num_files = len(file_list)
num_shards = (num_files + SAMPLES_PER_SHARD - 1) // SAMPLES_PER_SHARD

In [ ]:
for shard_idx in range(num_shards):

    start = shard_idx * SAMPLES_PER_SHARD
    end = min((shard_idx + 1) * SAMPLES_PER_SHARD, num_files)
    indices = list(range(start, end))

    # Optional: parallel read inside shard (safe here since it's small batch)
    with Pool(16) as p:  # adjust num workers if needed
        data = list(tqdm(p.imap(process_one, indices), total=len(indices)))

    shard_inputs, shard_targets = zip(*data)

    # Save shard
    shard_dict = {
        "inputs": torch.stack(shard_inputs),     # (N, 140, 11)
        "targets": torch.stack(shard_targets)    # (N, 1024, 1024)
    }

    shard_path = os.path.join(OUTPUT_DIR, f"shard-{shard_idx:05d}.pt")
    torch.save(shard_dict, shard_path)

    print(f"Saved shard {shard_idx+1}/{num_shards}: {shard_path}")